# Day 6: Apache Spark MLlib - Model Training
### IoT Anomaly & Fault Detection for EV Smart Charging Stations

This notebook demonstrates:
1. **Distributed Synthesized Telemetry:** Generating distributed training data in PySpark.
2. **VectorAssembler:** Packing scalar sensor features into contiguous dense vectors in RAM.
3. **StandardScaler:** Normalizing diverse sensor units (Volts, Amperes, Celsius).
4. **RandomForestClassifier:** Training an ensemble model to separate normal charging from electrical/thermal faults.
5. **Model Persistence:** Saving the fitted `PipelineModel` to disk for real-time streaming inference.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, rand
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import os

# Initialize local SparkSession
spark = SparkSession.builder \
    .appName("EV-Anomaly-Model-Training") \
    .master("local[2]") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark Version:", spark.version)

### Step 1: Synthesize Historical Telemetry Data
We generate 10,000 historical charger readings where ~10% represent overheating or electrical surge faults.

In [ ]:
df_raw = spark.range(0, 10000).withColumnRenamed("id", "row_id")
df_sim = df_raw.withColumn("is_fault", when(rand(seed=42) < 0.10, 1).otherwise(0))

# Normal: Temp 30-50 C, Volt 380-420 V, Current 30-100 A
# Fault : Temp 80-110 C, Volt 280-340 V, Current 130-180 A
df_data = df_sim.withColumn(
    "temperature_c",
    when(col("is_fault") == 1, 80.0 + rand(seed=101) * 30.0)
    .otherwise(30.0 + rand(seed=102) * 20.0)
).withColumn(
    "voltage_v",
    when(col("is_fault") == 1, 280.0 + rand(seed=201) * 60.0)
    .otherwise(380.0 + rand(seed=202) * 40.0)
).withColumn(
    "current_a",
    when(col("is_fault") == 1, 130.0 + rand(seed=301) * 50.0)
    .otherwise(30.0 + rand(seed=302) * 70.0)
).withColumn("label", col("is_fault").cast("double"))

df_data.select("temperature_c", "voltage_v", "current_a", "label").show(5)

### Step 2: Build the ML Pipeline
- `VectorAssembler`: Packs scalar columns into a contiguous vector array in memory for SIMD CPU operations.
- `StandardScaler`: Normalizes physical units.
- `RandomForestClassifier`: Fits ensemble decision trees.

In [ ]:
train_df, test_df = df_data.randomSplit([0.8, 0.2], seed=42)

assembler = VectorAssembler(
    inputCols=["voltage_v", "current_a", "temperature_c"],
    outputCol="raw_features"
)
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=10, maxDepth=5)

pipeline = Pipeline(stages=[assembler, scaler, rf])
model = pipeline.fit(train_df)

predictions = model.transform(test_df)
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

### Step 3: Persist Fitted Model to Disk
Save the PipelineModel to `/workspace/models/ev_anomaly_model` so our real-time streaming job can load it.

In [ ]:
model_path = "/workspace/models/ev_anomaly_model" if os.path.exists("/workspace") else "./models/ev_anomaly_model"
model.write().overwrite().save(model_path)
print(f"Model successfully persisted to: {model_path}")